In [ ]:
!nvidia-smi

In [ ]:
# Install zstd for faster decompression of datasets
!apt-get install -y zstd

# Install Ollama for running local LLMs
!curl -fsSL https://ollama.com/install.sh | sh

# Install the Ollama SDK
%pip install -q ollama

In [ ]:
import json
import subprocess
import time
import urllib.request

import requests
import ollama

MODEL = "qwen3.5:4b"

In [ ]:
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Wait until the server is ready (up to 15 seconds)
for _ in range(15):
    try:
        urllib.request.urlopen("http://localhost:11434")
        print("Ollama server is ready")
        break
    except Exception:
        time.sleep(1)
else:
    print("ERROR: Ollama server did not start. Try re-running this cell.")


In [ ]:
!ollama pull {MODEL}

In [ ]:
response = ollama.chat(
    model=MODEL,
    messages=[{"role": "user", "content": "What is the capital of Japan?"}],
    think=False,
)

print(response.message.content)


In [ ]:
response = ollama.chat(
    model=MODEL,
    messages=[{"role": "user", "content": "日本の首都はどこですか？"}],
    think=False,
)

print(response.message.content)


In [ ]:
response = ollama.chat(
    model=MODEL,
    messages=[
        {"role": "system", "content": "あなたは親切な日本語アシスタントです。必ず日本語で答えてください。"},
        {"role": "user",   "content": "What is the capital of France?"},
    ],
    think=False,
)

print(response.message.content)


In [ ]:
messages = [
    {"role": "system", "content": "あなたは親切な日本語アシスタントです。必ず日本語で答えてください。"},
]

turns = [
    "私の名前は太郎です。",
    "私の名前は何ですか？",
    "私の名前をローマ字で書いてください。",
]

for user_input in turns:
    messages.append({"role": "user", "content": user_input})
    print(f"User: {user_input}")

    response = ollama.chat(model=MODEL, messages=messages, think=False)

    assistant_reply = response.message.content
    messages.append({"role": "assistant", "content": assistant_reply})
    print(f"Assistant: {assistant_reply}\n")


In [ ]:
stream = ollama.chat(
    model=MODEL,
    messages=[
        {"role": "system", "content": "あなたは親切な日本語アシスタントです。必ず日本語で答えてください。"},
        {"role": "user",   "content": "日本の四季について説明してください。"},
    ],
    stream=True,
    think=False,
)

for chunk in stream:
    print(chunk.message.content, end="", flush=True)


In [ ]:
response = ollama.chat(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant that responds only in valid JSON."},
        {"role": "user", "content": (
            "日本の主要都市を3つ挙げて、以下のJSON形式で返してください:\n"
            '{"cities": [{"name": "...", "prefecture": "...", "population": ...}]}'
        )},
    ],
    format="json",
    think=False,
)

result = json.loads(response.message.content)
print(json.dumps(result, ensure_ascii=False, indent=2))


In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "指定した都市の現在の天気を取得する",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name in English (e.g. Tokyo, Osaka)"},
                    "unit": {"type": "string", "enum": ["celsius", "fahrenheit"], "description": "温度の単位"},
                },
                "required": ["city"],
            },
        },
    }
]

def get_weather(city, unit="celsius"):
    geo = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": city, "count": 1}
    ).json()
    if not geo.get("results"):
        return json.dumps({"error": f"City '{city}' not found."}, ensure_ascii=False)
    location = geo["results"][0]
    lat, lon = location["latitude"], location["longitude"]
    weather = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": lat, "longitude": lon,
            "current": "temperature_2m,weathercode,windspeed_10m",
            "temperature_unit": unit,
        }
    ).json()
    current = weather["current"]
    return json.dumps({
        "city": city, "temperature": current["temperature_2m"],
        "unit": unit, "windspeed_kmh": current["windspeed_10m"],
        "weathercode": current["weathercode"],
    }, ensure_ascii=False)

# Step 1: Send user message with tools
messages = [{"role": "user", "content": "東京の天気を教えてください。"}]
response = ollama.chat(model=MODEL, messages=messages, tools=tools, think=False)

messages.append(response.message)

# Step 2: Execute tool calls if requested
if response.message.tool_calls:
    for tool_call in response.message.tool_calls:
        args = tool_call.function.arguments  # already a dict in Ollama SDK
        print(f"Tool called: {tool_call.function.name}({args})")
        result = get_weather(**args)
        print(f"Tool result: {result}\n")
        messages.append({"role": "tool", "content": result})

    # Step 3: Send tool result back to get final response
    final_response = ollama.chat(model=MODEL, messages=messages, think=False)
    print(f"Assistant: {final_response.message.content}")
else:
    print(f"Assistant: {response.message.content}")


In [ ]:
messages = [
    {"role": "system", "content": "You are a haiku poet. Reply with exactly one haiku (3 lines only). No explanations, no titles, no romanization."},
    {"role": "user",   "content": "俳句を一句詠んでください。"},
]

configs = [
    {"label": "temperature=0.0 (決定的・一貫性重視)",  "options": {"temperature": 0.0}},
    {"label": "temperature=1.0 (バランス・デフォルト)", "options": {"temperature": 1.0}},
    {"label": "temperature=2.0 (ランダム・創造的)",    "options": {"temperature": 2.0}},
    {"label": "top_p=0.1 (保守的な候補に絞る)",        "options": {"temperature": 1.0, "top_p": 0.1}},
    {"label": "frequency_penalty=2.0 (繰り返し抑制)",  "options": {"temperature": 1.0, "frequency_penalty": 2.0}},
]

for config in configs:
    label = config.pop("label")
    response = ollama.chat(
        model=MODEL,
        messages=messages,
        think=False,
        **config,
    )
    print(f"[{label}]")
    print(response.message.content.strip())
    print()


In [ ]:
# ── 1. Router Agent ────────────────────────────────────────────────────────────
def router_agent(user_input: str) -> str:
    response = ollama.chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": (
                'You are a routing agent. Classify the user input into exactly one label. '
                'Reply ONLY with valid JSON: {"label": "<label>"} '
                'Labels: "weather" (weather questions), "haiku" (creative/poem requests), "general" (everything else).'
            )},
            {"role": "user", "content": user_input},
        ],
        format="json",
        think=False,
    )
    result = json.loads(response.message.content)
    print(f"[Router] → label: {result['label']}")
    return result["label"]

# ── 2. Specialist Agents ───────────────────────────────────────────────────────
def weather_specialist(user_input: str) -> str:
    tools = [{
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather for a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name in English"},
                },
                "required": ["city"],
            },
        },
    }]

    def get_weather(city):
        geo = requests.get("https://geocoding-api.open-meteo.com/v1/search",
                           params={"name": city, "count": 1}).json()
        if not geo.get("results"):
            return json.dumps({"error": f"City '{city}' not found."})
        loc = geo["results"][0]
        weather = requests.get("https://api.open-meteo.com/v1/forecast", params={
            "latitude": loc["latitude"], "longitude": loc["longitude"],
            "current": "temperature_2m,weathercode,windspeed_10m",
        }).json()
        c = weather["current"]
        return json.dumps({"city": city, "temperature_celsius": c["temperature_2m"],
                           "windspeed_kmh": c["windspeed_10m"]})

    messages = [{"role": "user", "content": user_input}]
    res = ollama.chat(model=MODEL, messages=messages, tools=tools, think=False)
    messages.append(res.message)

    if res.message.tool_calls:
        for tc in res.message.tool_calls:
            args = tc.function.arguments  # already a dict
            tool_result = get_weather(**args)
            print(f"[Weather Specialist] tool call: get_weather({args})")
            messages.append({"role": "tool", "content": tool_result})
        final = ollama.chat(model=MODEL, messages=messages, think=False)
        return final.message.content
    return res.message.content

def haiku_specialist(user_input: str) -> str:
    res = ollama.chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a haiku poet. Reply with exactly one haiku (3 lines only). No explanations."},
            {"role": "user", "content": user_input},
        ],
        think=False,
    )
    return res.message.content

def general_specialist(user_input: str) -> str:
    res = ollama.chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful assistant. Answer concisely in Japanese."},
            {"role": "user", "content": user_input},
        ],
        think=False,
    )
    return res.message.content

SPECIALISTS = {
    "weather": weather_specialist,
    "haiku":   haiku_specialist,
    "general": general_specialist,
}

# ── 3. Guardrail Agent ─────────────────────────────────────────────────────────
def guardrail_agent(specialist_output: str) -> str:
    res = ollama.chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": (
                "You are a politeness guardrail. Rewrite the given response to be warm, "
                "polite, and natural in Japanese. Keep the content accurate. Output the rewritten response only."
            )},
            {"role": "user", "content": specialist_output},
        ],
        think=False,
    )
    return res.message.content

# ── Run the pipeline ───────────────────────────────────────────────────────────
def run_agent(user_input: str):
    print(f"User: {user_input}\n")
    label    = router_agent(user_input)
    raw      = SPECIALISTS.get(label, general_specialist)(user_input)
    print(f"[Specialist ({label})]\n{raw}\n")
    polished = guardrail_agent(raw)
    print(f"[Guardrail] Final response:\n{polished}")

run_agent("東京の天気を教えてください。")
# run_agent("秋について俳句を詠んでください。")
# run_agent("日本で一番高い山は何ですか？")